## hpi

In [1]:
import pandas as pd
import re

# Read the data
cholecystitis_df = pd.read_csv('raw_data/diverticulitis_hadm_info_first_diag.csv')


def parse_patient_history(text):
    """Extract structured sections from patient history text"""
    if pd.isna(text):
        return {
            'hpi': None,
            'past_medical_history': None,
            'past_surgical_history': None,
            'social_history': None,
            'family_history': None
        }
    
    # Initialize dict
    sections = {}
    
    # Extract Past Medical History (two patterns: "PMH:" and "Past Medical History:")
    pmh_match = re.search(r'Past Medical History:\s*(.+?)(?=\s*PSH:|Social History:|Family History:|$)', text, re.DOTALL)
    if not pmh_match:
        pmh_match = re.search(r'PMH:\s*(.+?)(?=\s*PSH:|Social History:|Family History:|$)', text, re.DOTALL)
    sections['past_medical_history'] = pmh_match.group(1).strip() if pmh_match else None
    
    # Extract Past Surgical History
    psh_match = re.search(r'PSH:\s*(.+?)(?=\s*Social History:|Family History:|$)', text, re.DOTALL)
    sections['past_surgical_history'] = psh_match.group(1).strip() if psh_match else None
    
    # Extract Social History
    social_match = re.search(r'Social History:\s*(.+?)(?=\s*Family History:|$)', text, re.DOTALL)
    sections['social_history'] = social_match.group(1).strip() if social_match else None
    
    # Extract Family History
    family_match = re.search(r'Family History:\s*(.+?)$', text, re.DOTALL)
    sections['family_history'] = family_match.group(1).strip() if family_match else None
    
    # Extract HPI (everything before first section header)
    hpi_match = re.search(r'^(.+?)(?=\s*Past Medical History:|PMH:|Social History:|Family History:|$)', text, re.DOTALL)
    sections['hpi'] = hpi_match.group(1).strip() if hpi_match else text.strip()
    
    return sections

# Apply parsing
history_parsed = cholecystitis_df['Patient History'].apply(parse_patient_history)
history_df = pd.DataFrame(history_parsed.tolist())
history_df['hadm_id'] = cholecystitis_df['hadm_id']

In [2]:
history_df.head()

,past_medical_history,past_surgical_history,social_history,family_history,hpi,hadm_id
0,Neuropathy Insomnia Hypercholesteremia Hyperte...,None,___,Mother had a large MI at age ___ and died from...,Patient is a ___ M with PMHx of atrial fibrill...,24911566
1,1. Coronary artery disease status post coronar...,None,___ ___ History: Mother - CHF Father - died ...,None,This is a ___ yo M with late stage Alzheimer's...,24600926
2,Past Medical History: Headaches,None,___,Family History: Mother with HTN. Father died a...,"___ presenting with 5 days of LLQ, subjective ...",28694648
3,Hand surgery,None,___,___ contributory,This is a ___ year old male who presented to t...,20410636
4,"PMH: IBS type symptoms, recently improved. Po...","lap ccy, c-scope ___ (Multiple mild non-bleedi...",___,Noncontributory,"___ female with h/o IBS (chronic diarrhea, blo...",23877579


## Lab test (textual)

In [3]:
import pandas as pd
import json
import re

def extract_numeric_value(value_str):
    """
    Extract numeric value from string like "62.0 IU/L", "1.0 #/hpf", etc.
    Returns None if value is text-only (NEG., NONE, etc.)
    """
    if pd.isna(value_str) or value_str is None:
        return None, None
    
    value_str = str(value_str).strip()
    
    # Check if it's a text-only result (NEG., NONE, HOLD., etc.)
    text_only_patterns = ['NEG', 'NONE', 'HOLD', 'DISCARD', 'FEW', 'RARE', 
                          'MODERATE', 'MANY', 'Clear', 'Yellow', 'NotDone',
                          'RANDOM', 'Using this', '<', '>']
    
    for pattern in text_only_patterns:
        if pattern in value_str:
            return None, value_str  # Return original text
    
    # Try to extract numeric value
    match = re.search(r'([\d.]+)', value_str)
    if match:
        try:
            numeric_val = float(match.group(1))
            return numeric_val, value_str
        except ValueError:
            return None, value_str
    
    return None, value_str

def classify_lab_result(test_value, ref_lower, ref_upper):
    """
    Classify lab result as:
    - Normal: Normal (within range)
    - Decreased: Too low (below lower limit)
    - Elevated: Too high (above upper limit)
    - N/A: Keep original text if non-numeric
    """
    numeric_val, original_text = extract_numeric_value(test_value)
    
    # If no numeric value extracted, return original text
    if numeric_val is None:
        return original_text if original_text else "N/A"
    
    # If no reference ranges available, return "unknown"
    if pd.isna(ref_lower) and pd.isna(ref_upper):
        return "unknown"
    
    # Compare with reference ranges
    if not pd.isna(ref_lower) and numeric_val < ref_lower:
        return "Decreased"  # Too low
    elif not pd.isna(ref_upper) and numeric_val > ref_upper:
        return "Elevated"   # Too high
    else:
        return "Normal"   # Normal

def process_lab_tests(row):
    """
    Process laboratory tests for one patient
    Returns a dictionary of test_id: classification
    """
    try:
        lab_tests = json.loads(row['Laboratory Tests']) if isinstance(row['Laboratory Tests'], str) else row['Laboratory Tests']
        ref_lower = json.loads(row['Reference Range Lower']) if isinstance(row['Reference Range Lower'], str) else row['Reference Range Lower']
        ref_upper = json.loads(row['Reference Range Upper']) if isinstance(row['Reference Range Upper'], str) else row['Reference Range Upper']
    except:
        return {}
    
    if not isinstance(lab_tests, dict):
        return {}
    
    classified_results = {}
    
    for test_id, test_value in lab_tests.items():
        lower = ref_lower.get(test_id) if isinstance(ref_lower, dict) else None
        upper = ref_upper.get(test_id) if isinstance(ref_upper, dict) else None
        
        classification = classify_lab_result(test_value, lower, upper)
        classified_results[test_id] = classification
    
    return classified_results


In [4]:
# Example usage:
df = pd.read_csv('raw_data/diverticulitis_hadm_info_first_diag.csv')

# Apply processing
df['lab_tests_classified'] = df.apply(process_lab_tests, axis=1)

# Convert to JSON string for storage
df['lab_tests_classified_json'] = df['lab_tests_classified'].apply(
    lambda x: json.dumps(x) if isinstance(x, dict) else "{}")


lab_classification_df = df[['hadm_id', 'lab_tests_classified_json']].copy()
lab_classification_df.columns = ['hadm_id', 'lab_tests_classified_text']

In [5]:
lab_classification_df.head()

,hadm_id,lab_tests_classified_text
0,24911566,"{""51146"": ""Normal"", ""51006"": ""Elevated"", ""5098..."
1,24600926,"{""50861"": ""Normal"", ""51200"": ""Normal"", ""51221""..."
2,28694648,"{""50933"": ""HOLD. DISCARD GREATER THAN 4 HOURS..."
3,20410636,"{""50861"": ""Normal"", ""51233"": ""NORMAL."", ""51244..."
4,23877579,"{""50861"": ""Normal"", ""51279"": ""Normal"", ""51277""..."


### Replace test code by name

In [7]:
import pandas as pd
import json
import ast

# ---------------- paths ----------------
mapping_path = "raw_data/lab_test_mapping.csv"

# 1) Load mapping and keep only exact item-level rows
map_df= pd.read_csv(mapping_path)

# Drop rows without itemid or label (these are panel/group rows like BMP/CMP)
map_df = map_df.dropna(subset=["itemid", "label"]).copy()
map_df["itemid"] = map_df["itemid"].astype(int)
map_df["label"] = map_df["label"].astype(str).str.strip()

# If duplicate itemid exists, keep first label (you can change to 'last' if preferred)
map_df = map_df.drop_duplicates(subset=["itemid"], keep="first")

# Exact mapping: itemid -> label
id_to_label = dict(zip(map_df["itemid"], map_df["label"]))

# 2) Load extracted sequences
df_lab_tests= lab_classification_df

def parse_lab_dict(x):
    if pd.isna(x):
        return {}
    s = str(x)
    try:
        return json.loads(s)          # preferred
    except Exception:
        try:
            return ast.literal_eval(s) # fallback
        except Exception:
            return {}

def replace_ids_with_exact_names(d):
    new_d = {}
    for k, v in d.items():
        try:
            itemid = int(k)
            # exact mapping only; keep original id string if not found
            new_key = id_to_label.get(itemid, str(k))
        except Exception:
            new_key = str(k)

        # avoid overwriting if same name appears multiple times
        if new_key in new_d:
            if not isinstance(new_d[new_key], list):
                new_d[new_key] = [new_d[new_key]]
            new_d[new_key].append(v)
        else:
            new_d[new_key] = v
    return new_d

# 3) Replace keys in lab_tests_classified
parsed = df_lab_tests["lab_tests_classified_text"].apply(parse_lab_dict)
mapped = parsed.apply(replace_ids_with_exact_names)

# overwrite the original column with mapped JSON string
df_lab_tests["lab_tests_classified_text"] = mapped.apply(lambda d: json.dumps(d, ensure_ascii=False))


In [9]:
df_lab_tests

,hadm_id,lab_tests_classified_text
0,24911566,"{""Basophils"": ""Normal"", ""Urea Nitrogen"": ""Elev..."
1,24600926,"{""Alanine Aminotransferase (ALT)"": ""Normal"", ""..."
2,28694648,"{""Green Top Hold, plasma"": ""HOLD. DISCARD GRE..."
3,20410636,"{""Alanine Aminotransferase (ALT)"": ""Normal"", ""..."
4,23877579,"{""Alanine Aminotransferase (ALT)"": ""Normal"", ""..."
...,...,...
252,26360008,"{""Alanine Aminotransferase (ALT)"": ""Normal"", ""..."
253,21055588,"{""Green Top Hold, plasma"": ""HOLD. DISCARD GRE..."
254,22255808,"{""INR(PT)"": ""Normal"", ""Alanine Aminotransferas..."
255,24921121,"{""Alanine Aminotransferase (ALT)"": ""Normal"", ""..."


## Radiology sequences

In [10]:
df_hpi_lab = df_lab_tests.merge(history_df, on='hadm_id', how='left')

In [11]:
df_hpi_lab

,hadm_id,lab_tests_classified_text,past_medical_history,past_surgical_history,social_history,family_history,hpi
0,24911566,"{""Basophils"": ""Normal"", ""Urea Nitrogen"": ""Elev...",Neuropathy Insomnia Hypercholesteremia Hyperte...,None,___,Mother had a large MI at age ___ and died from...,Patient is a ___ M with PMHx of atrial fibrill...
1,24600926,"{""Alanine Aminotransferase (ALT)"": ""Normal"", ""...",1. Coronary artery disease status post coronar...,None,___ ___ History: Mother - CHF Father - died ...,None,This is a ___ yo M with late stage Alzheimer's...
2,28694648,"{""Green Top Hold, plasma"": ""HOLD. DISCARD GRE...",Past Medical History: Headaches,None,___,Family History: Mother with HTN. Father died a...,"___ presenting with 5 days of LLQ, subjective ..."
3,20410636,"{""Alanine Aminotransferase (ALT)"": ""Normal"", ""...",Hand surgery,None,___,___ contributory,This is a ___ year old male who presented to t...
4,23877579,"{""Alanine Aminotransferase (ALT)"": ""Normal"", ""...","PMH: IBS type symptoms, recently improved. Po...","lap ccy, c-scope ___ (Multiple mild non-bleedi...",___,Noncontributory,"___ female with h/o IBS (chronic diarrhea, blo..."
...,...,...,...,...,...,...,...
252,26360008,"{""Alanine Aminotransferase (ALT)"": ""Normal"", ""...",PAST MEDICAL HISTORY: IBS HA Psoriasis Osteoar...,None,___,"Father- colon cancer Mother- arthritis, asthma...",Pt has several months of severe abdominal pain...
253,21055588,"{""Green Top Hold, plasma"": ""HOLD. DISCARD GRE...",PMH: ___'s thyroiditis,None,___,"No first or second degree FH of GI cancers, or...",___ male past medical history significant for ...
254,22255808,"{""INR(PT)"": ""Normal"", ""Alanine Aminotransferas...",PAST MEDICAL HISTORY: right trochanteric bursi...,None,___,Non-contributory,___ with history of IBS who presents with 3-da...
255,24921121,"{""Alanine Aminotransferase (ALT)"": ""Normal"", ""...",-Non-ischemic cardiomyopathy -Mitral and tricu...,None,___,+CAD. Both parents died in fire age <___.,Ms. ___ is a ___ year old female with PMH Grav...


In [12]:
import pandas as pd
import json
import ast
from collections import defaultdict

# ---------------- paths ----------------
# If your lab test IDs are already replaced with exact names, point this to that file.
radio_path = "raw_data/diverticulitis_hadm_info_first_diag.csv"
out_path = "state_text_diver.json"

# ---------------- helpers ----------------
def parse_obj(x):
    """Parse JSON-ish string to python object."""
    if pd.isna(x):
        return None
    s = str(x).strip()
    if s == "" or s.lower() == "nan":
        return None
    try:
        return json.loads(s)
    except Exception:
        try:
            return ast.literal_eval(s)
        except Exception:
            return s  # keep raw text if not parseable

def build_radiology_sequence(radiology_obj):
    """
    Preserve original radiology test order as a list of events.
    Output example:
    [
      {"modality": "CT", "note_id": ..., "region": ..., "exam_name": ..., "report": ...},
      {"modality": "Ultrasound", ...},
      ...
    ]
    """
    out = []

    if radiology_obj is None:
        return out

    def normalize_item(item):
        if isinstance(item, dict):
            modality = str(item.get("Modality", "other")).strip() or "other"
            return {
                "modality": modality,
                "note_id": item.get("Note ID"),
                "region": item.get("Region"),
                "exam_name": item.get("Exam Name"),
                "report": item.get("Report"),
            }
        return {"modality": "other", "raw": item}

    if isinstance(radiology_obj, list):
        for item in radiology_obj:
            out.append(normalize_item(item))
    elif isinstance(radiology_obj, dict):
        out.append(normalize_item(radiology_obj))
    else:
        out.append({"modality": "other", "raw": radiology_obj})

    return out

# ---------------- load ----------------
rad_df = pd.read_csv(radio_path)

# normalize hadm_id
df_hpi_lab["hadm_id"] = pd.to_numeric(df_hpi_lab["hadm_id"], errors="coerce").astype("Int64")
rad_df["hadm_id"] = pd.to_numeric(rad_df["hadm_id"], errors="coerce").astype("Int64")

df_hpi_lab = df_hpi_lab.dropna(subset=["hadm_id"]).copy()
rad_df = rad_df.dropna(subset=["hadm_id"]).copy()

# keep only needed columns
seq_small = df_hpi_lab[["hadm_id", "hpi", "lab_tests_classified_text"]].copy()
rad_small = rad_df[["hadm_id", "Radiology"]].copy()

# if duplicate hadm_id exists, keep first row (change if you need different behavior)
seq_small = seq_small.drop_duplicates(subset=["hadm_id"], keep="first")
rad_small = rad_small.drop_duplicates(subset=["hadm_id"], keep="first")

# merge
merged = seq_small.merge(rad_small, on="hadm_id", how="left")

# ---------------- build output ----------------
state = {}

for _, row in merged.iterrows():
    hadm = str(int(row["hadm_id"]))

    hpi_text = "" if pd.isna(row["hpi"]) else str(row["hpi"])

    lab_obj = parse_obj(row["lab_tests_classified_text"])
    if not isinstance(lab_obj, dict):
        lab_obj = {} if lab_obj is None else {"raw": lab_obj}

    radiology_obj = parse_obj(row["Radiology"])
    radiology_seq = build_radiology_sequence(radiology_obj)

    state[hadm] = {
        "hpi": hpi_text,
        "lab_tests": lab_obj,
        "radiology": radiology_seq
    }

# save
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(state, f, ensure_ascii=False, indent=2)

print(f"Saved {len(state)} patients to {out_path}")

Saved 257 patients to state_text_diver.json


## Dual-Core

In [19]:
import json

path = "data_chole/state_trajectories_denoised_chole.json"

with open(path, "r") as f:
    data = json.load(f)

# patient_id -> first test after Lab_Panel (or None if not available)
first_test_after_lab = {}

for pid, traj in data.items():
    test = None
    for state in traj:  # go in time order
        hist = state.get("Modality_History", [])
        if len(hist) >= 2 and hist[0] == "Lab_Panel":
            test = hist[1]
            break
    first_test_after_lab[pid] = test

# all possible first radiology tests (drop None)
first_radiology_possibilities = set(t for t in first_test_after_lab.values() if t is not None)

print(first_radiology_possibilities)
# optional: print(first_test_after_lab)

{'ERCP_Abdomen', 'Ultrasound_Abdomen', 'MRCP_Abdomen', 'Drainage_Abdomen', 'Radiograph_Abdomen', 'CT_Head', 'CT_Chest', 'Ultrasound_Neck', 'CTU_Abdomen', 'CT_Abdomen', 'Radiograph_Chest', 'MRI_Abdomen'}


### extract patient features (for retrieval)

In [20]:
import json
import csv
import re
# ──────────────────────────────────────────────────────────────
# 1. Load data sources
# ──────────────────────────────────────────────────────────────
with open('data_chole/state_text_chole.json') as f:
    state_text = json.load(f)
with open('raw_data/cholecystitis_hadm_info_first_diag.csv') as f:
    reader = csv.DictReader(f)
    raw_rows = {row['hadm_id']: row for row in reader}

In [13]:
# ──────────────────────────────────────────────────────────────
# 2. GROUP 3: Lab features  (state_text_chole → already Normal/Elevated/Decreased)
# ──────────────────────────────────────────────────────────────
LAB_MAP = {
    'wbc_status':        ['White Blood Cells', 'WBC'],
    'bilirubin_status':  ['Bilirubin, Total'],
    'alt_status':        ['Alanine Aminotransferase (ALT)'],
    'alkphos_status':    ['Alkaline Phosphatase'],
    'lipase_status':     ['Lipase'],
    'creatinine_status': ['Creatinine'],
    'inr_status':        ['INR(PT)'],
    'lactate_status':    ['Lactate'],
}
STATUS_ENCODE = {'Elevated': 1, 'Normal': 0, 'Decreased': -1}
def encode_lab(raw_value):
    """Handle single string or list (multiple draws → take max severity)."""
    if isinstance(raw_value, list):
        vals = [STATUS_ENCODE[v] for v in raw_value if v in STATUS_ENCODE]
        return max(vals) if vals else 0
    return STATUS_ENCODE.get(raw_value, 0)
def extract_labs(pid):
    labs = state_text.get(pid, {}).get('lab_tests', {})
    result = {}
    for feat, keys in LAB_MAP.items():
        raw = next((labs[k] for k in keys if k in labs), None)
        result[feat] = encode_lab(raw) if raw is not None else 0
    return result


In [21]:
# ──────────────────────────────────────────────────────────────
# 3. GROUP 1: Pain profile  (HPI text → regex/keyword)
# ──────────────────────────────────────────────────────────────
LOCATION_PATTERNS = [
    ('RUQ',              r'\bruq\b|right upper quadrant'),
    ('RLQ',              r'\brlq\b|right lower quadrant'),
    ('LLQ',              r'\bllq\b|left lower quadrant'),
    ('LUQ',              r'\bluq\b|left upper quadrant'),
    ('Epigastric',       r'\bepigastric\b|epigastrium'),
    ('Periumbilical',    r'\bperiumbilical\b|peri.umbilical|umbilical region'),
    ('Pelvic',           r'\bpelvic\b|suprapubic'),
    ('General_Abdomen',  r'\bdiffuse\b|generalized abdom|entire abdom|pan.?abdominal'),
]
def extract_pain_location(text):
    t = text.lower()
    for loc, pattern in LOCATION_PATTERNS:
        if re.search(pattern, t):
            return loc
    return 'General_Abdomen'
def extract_hpi_features(hpi):
    t = hpi.lower()
    return {
        'pain_location':   extract_pain_location(hpi),
        'pain_onset':      1 if re.search(r'\bsudden\b|sudden onset|acute onset|abrupt', t) else 0,
        'nausea_vomiting': 1 if re.search(r'\bnausea\b|\bvomit\b|\bemesis\b|\bn/v\b', t) else 0,
        'subj_fever':      1 if re.search(r'\bfever\b|\bfevers\b|\bchills\b|\bfebrile\b', t) else 0,
        'jaundice':        1 if re.search(r'\bjaundice\b|\bicteric\b|\byellow.*skin\b|scleral.*icteric', t) else 0,
        'radiating_pain':  1 if re.search(r'\bradiating\b|\bradiated\b|\bradiate\b', t) else 0,
    }


In [28]:
# ──────────────────────────────────────────────────────────────
# 4. GROUP 2: Physical signs  (PE text → vitals regex + keywords)
# ──────────────────────────────────────────────────────────────
def parse_vitals(pe_text):
    """
    Handles labeled format:   T: 98.4  HR: 72  BP: 134/90
    And positional format:    Vitals: 98.8 73 153/99 15 100% RA  (T HR BP RR O2)
    Returns {'temp_f', 'hr', 'sbp'}, each None if not found.
    """
    temp_f = hr = sbp = None
    # --- Labeled format (T: / HR: / BP:) ---
    m = re.search(r'\bT[: ]*(\d{2,3}(?:\.\d)?)\b', pe_text)
    if m:
        temp_f = float(m.group(1))
    m = re.search(r'\bHR[: ]+(\d{2,3})\b', pe_text)
    if m:
        hr = int(m.group(1))
    m = re.search(r'\bBP[: ]+(\d{2,3})/\d{2,3}\b', pe_text)
    if m:
        sbp = int(m.group(1))
    # --- Positional fallback: "Vitals: T HR BP RR O2" ---
    if temp_f is None:
        vm = re.search(r'[Vv]itals?[: ]+([^\n]+)', pe_text)
        if vm:
            line = vm.group(1)
            bp_m = re.search(r'(\d{2,3})/\d{2,3}', line)
            if bp_m:
                sbp = sbp or int(bp_m.group(1))
                pre = line[:bp_m.start()]
                # Temperature: first float between 95–105
                t_cands = re.findall(r'\b(9[5-9](?:\.\d)?|10[0-4](?:\.\d)?)\b', pre)
                if t_cands:
                    temp_f = float(t_cands[0])
                # HR: first 2-3 digit number 40–200 that isn't the temperature
                for n in re.findall(r'\b(\d{2,3})\b', pre):
                    nf = float(n)
                    if 40 <= nf <= 200 and not (95 <= nf <= 105):
                        hr = hr or int(n)
                        break
    return {'temp_f': temp_f, 'hr': hr, 'sbp': sbp}

def has_sign_without_negation(text, pattern):
    """Return 1 only if pattern found AND not preceded by negation within 4 words."""
    for m in re.finditer(pattern, text):
        # Look at up to 40 chars before the match
        pre = text[max(0, m.start()-40): m.start()]
        if not re.search(r'\bno\b|\bnot\b|\bwithout\b|\bdenies\b|\bdenied\b', pre):
            return 1
    return 0



def extract_pe_features(pe_text):
    t = pe_text.lower()
    v = parse_vitals(pe_text)
    return {
        'fever_objective':  1 if (v['temp_f'] and v['temp_f'] > 100.4) else 0,
        'tachycardia':      1 if (v['hr']     and v['hr']     > 100)   else 0,
        'hypotension':      1 if (v['sbp']    and v['sbp']    < 90)    else 0,
        'peritoneal_signs': has_sign_without_negation(
    t, r'\bguarding\b|\brigidity\b|\brebound\b'),
        'murphy_sign':      1 if re.search(r"\bmurphy'?s?\b", t) else 0,
    }


In [29]:
# ──────────────────────────────────────────────────────────────
# 5. GROUP 4: Risk flags  (HPI text → keyword extraction)
# ──────────────────────────────────────────────────────────────
def extract_age_group(hpi_text, lab_text=''):
    """
    MIMIC anonymizes ages as '___'.
    Best proxy: GFR report embeds 'age group XX-YY' in the lab text.
    Falls back to 1 (middle-aged) if not found.
    """
    m = re.search(r'age group (\d+)-(\d+)', lab_text)
    if m:
        mid = (int(m.group(1)) + int(m.group(2))) / 2
        return 0 if mid < 40 else (2 if mid > 65 else 1)
    return 1
def extract_risk_flags(hpi_text, lab_text=''):
    t = hpi_text.lower()
    return {
        'has_malignancy':         1 if re.search(
            r'\bcancer\b|\btumor\b|\bmalignancy\b|\blymphoma\b|\bleukemia\b'
            r'|\bmyeloma\b|\bcarcinoma\b|\bhcc\b|\bmetastas', t) else 0,
        'is_immunosuppressed':    1 if re.search(
            r'\bchemotherapy\b|\bchemo\b|\bimmunosuppress\b|\btransplant\b'
            r'|\bcorticosteroid\b|\bsteroids?\b|\bhiv\b|\baids\b'
            r'|\bdecitabine\b|\btacrolimus\b|\bmycophenolate\b', t) else 0,
        'recent_hospitalization': 1 if re.search(
            r'recently (admitted|discharged|hospitalized)'
            r'|recent (admission|discharge)|transferred from'
            r'|last week.*admitted|admitted.*last week', t) else 0,
        'recent_procedure':       1 if re.search(
    r'\bercp\b|\bstent\b|post.?op|post.?surg'
    r'|recent.*(?:surgery|procedure|operation)'
    r'|(?:surgery|procedure|operation).*(?:last week|last month|recent)', t) else 0,
        'has_diabetes':           1 if re.search(
            r'\bdiabetes\b|\bdiabetic\b|\bt2dm\b|\bt1dm\b'
            r'|\binsulin.dependent\b', t) else 0,
        'age_group':              extract_age_group(hpi_text, lab_text),
    }

In [30]:
# ──────────────────────────────────────────────────────────────
# 6. Main: combine all groups per patient
# ──────────────────────────────────────────────────────────────
def extract_patient_features(pid):
    pid = str(pid)
    hpi_text  = state_text.get(pid, {}).get('hpi', '')
    raw_row   = raw_rows.get(pid, {})
    pe_text   = raw_row.get('Physical Examination', '')
    lab_text  = raw_row.get('Laboratory Tests', '')   # raw text, for age proxy only
    features = {'hadm_id': pid}
    features.update(extract_hpi_features(hpi_text))       # Group 1  (6 features)
    features.update(extract_pe_features(pe_text))          # Group 2  (5 features)
    features.update(extract_labs(pid))                     # Group 3  (8 features)
    features.update(extract_risk_flags(hpi_text, lab_text))# Group 4  (6 features)
    return features
# Extract all patients
patient_features = {pid: extract_patient_features(pid) for pid in state_text.keys()}
print(f"Extracted features for {len(patient_features)} patients")

Extracted features for 648 patients


In [31]:
# ── Quick QA ──────────────────────────────────────────────────
import collections
# Count feature coverage (% non-zero or non-default)
all_feats = list(next(iter(patient_features.values())).keys())
all_feats.remove('hadm_id')
coverage = {}
for feat in all_feats:
    nonzero = sum(1 for p in patient_features.values() if p[feat] != 0)
    coverage[feat] = nonzero / len(patient_features)
print("\nFeature activation rates:")
for feat, rate in sorted(coverage.items(), key=lambda x: -x[1]):
    print(f"  {feat:<28} {rate:.1%}")


Feature activation rates:
  pain_location                100.0%
  nausea_vomiting              86.1%
  age_group                    83.5%
  subj_fever                   78.2%
  wbc_status                   55.4%
  inr_status                   38.4%
  alt_status                   33.8%
  alkphos_status               29.3%
  radiating_pain               19.8%
  bilirubin_status             17.9%
  lactate_status               16.4%
  creatinine_status            16.2%
  recent_procedure             16.2%
  pain_onset                   13.6%
  lipase_status                13.1%
  peritoneal_signs             9.6%
  jaundice                     9.4%
  tachycardia                  5.4%
  is_immunosuppressed          4.8%
  has_malignancy               4.5%
  recent_hospitalization       3.7%
  murphy_sign                  3.2%
  has_diabetes                 2.8%
  fever_objective              1.9%
  hypotension                  0.5%


In [32]:
import json
import csv

# ── Save as JSON (preserves all types) ────────────────────────
with open('data_chole/patient_features_chole.json', 'w') as f:
    json.dump(patient_features, f, indent=2)

# ── Save as CSV (easy inspection) ─────────────────────────────
fieldnames = list(next(iter(patient_features.values())).keys())
with open('data_chole/patient_features_chole.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(patient_features.values())

print(f"Saved {len(patient_features)} patients to:")
print("  data_chole/patient_features_chole.json")
print("  data_chole/patient_features_chole.csv")

Saved 648 patients to:
  data_chole/patient_features_chole.json
  data_chole/patient_features_chole.csv


### Objective 1: Rubric vs empirical

In [35]:
import json

# ──────────────────────────────────────────────────────────────
# 1. Load state trajectories (for rubric input + actual sequence)
# ──────────────────────────────────────────────────────────────
with open('data_chole/state_trajectories_denoised_chole.json') as f:
    trajectories = json.load(f)


# ──────────────────────────────────────────────────────────────
# 2. Rubric Engine
#    Input:  initial state (state[0] from trajectory)
#    Output: recommended first radiology test
# ──────────────────────────────────────────────────────────────
def rubric_recommend(state):
    topo    = state['Topographic_Focus']
    ss      = state['Systemic_Stress']
    liver   = state['Organ_Stress_Index']['Liver']
    kidney  = state['Organ_Stress_Index']['Kidney']
    pancreas= state['Organ_Stress_Index']['Pancreas']
    stone   = state['Structural_Findings']['stone_detected']
    wall    = state['Structural_Findings']['wall_thickening']
    fluid   = state['Structural_Findings']['fluid_collection']
    dilation= state['Structural_Findings']['organ_dilation']

    if topo == 'Chest':
        return 'CT_Chest' if ss > 0.5 else 'Radiograph_Chest'

    elif topo in ('RUQ', 'LUQ'):
        if dilation == 1:
            return 'MRCP_Abdomen'
        elif stone == 1 or liver > 0.3:
            return 'Ultrasound_Abdomen'
        elif ss > 0.5 or wall == 1:
            return 'CT_Abdomen'
        else:
            return 'Ultrasound_Abdomen'

    elif topo == 'Epigastric':
        if pancreas > 0.3 and stone == 1:
            return 'Ultrasound_Abdomen'
        else:
            return 'CT_Abdomen'

    elif topo in ('RLQ', 'Periumbilical'):
        return 'CT_Abdomen' if (ss > 0.5 or fluid == 1) else 'Ultrasound_Abdomen'

    elif topo == 'LLQ':
        return 'CT_Abdomen'

    elif topo == 'Pelvic':
        return 'CTU_Abdomen' if kidney > 0.3 else 'Ultrasound_Abdomen'

    else:  # General_Abdomen, Other
        if ss > 0.5 or fluid == 1:
            return 'CT_Abdomen'
        elif stone == 1:
            return 'Ultrasound_Abdomen'
        elif wall == 1:
            return 'CT_Abdomen'
        else:
            return 'Radiograph_Abdomen'


# ──────────────────────────────────────────────────────────────
# 3. Alignment Cost Matrix
#    Operationalizes "how wrong" a deviation is.
#    Anatomy:  sync move = 0 cost
#              log move (patient did Y, rubric didn't expect Y) = log cost
#              model move (rubric expected X, patient skipped X) = model cost
# ──────────────────────────────────────────────────────────────
MODALITY_GROUP = {
    'CT_Abdomen':       'abdomen',
    'Ultrasound_Abdomen': 'abdomen',
    'MRCP_Abdomen':     'abdomen',
    'MRI_Abdomen':      'abdomen',
    'CTU_Abdomen':      'abdomen',
    'Radiograph_Abdomen': 'abdomen',
    'ERCP_Abdomen':     'abdomen',
    'Drainage_Abdomen': 'abdomen',
    'CT_Chest':         'chest',
    'Radiograph_Chest': 'chest',
    'CT_Head':          'head',
    'Ultrasound_Neck':  'neck',
}

def alignment_cost(expected, actual):
    """
    Returns (cost, move_type) for a single-step comparison.
    
    Move types (standard PM alignment terminology):
      'sync'        – expected == actual, no deviation
      'wrong_mod'   – same body region, different modality (partial deviation)
      'wrong_region'– different body region (full deviation)

    Cost scale: 0 = perfect, 1 = partial, 2 = full mismatch
    Fitness = 1 - cost / MAX_COST  where MAX_COST = 2
    """
    MAX_COST = 2
    if expected == actual:
        return 0, 'sync'
    g_exp = MODALITY_GROUP.get(expected, 'unknown')
    g_act = MODALITY_GROUP.get(actual,   'unknown')
    if g_exp == g_act:
        cost, move = 1, 'wrong_mod'
    else:
        cost, move = 2, 'wrong_region'
    return cost, move

def fitness_from_cost(cost, max_cost=2):
    return round(1 - cost / max_cost, 4)


# ──────────────────────────────────────────────────────────────
# 4. Conformance Checking Loop
#    For each patient:
#      - initial_state  = trajectory[0]          (features before any radiology)
#      - expected_test  = rubric(initial_state)
#      - actual_test    = trajectory[1]['Modality_History'][1]
#      - fitness        = alignment fitness score
# ──────────────────────────────────────────────────────────────
conformance_results = {}

for pid, states in trajectories.items():
    initial_state = states[0]

    # Rubric recommendation
    expected_test = rubric_recommend(initial_state)

    # Actual first radiology test (state[1] was guaranteed to exist above)
    if len(states) > 1:
        actual_test = states[1]['Modality_History'][1]  # second element = first radiology
    else:
        actual_test = None  # no radiology performed

    if actual_test is None:
        cost, move_type, fitness = None, 'no_radiology', None
    else:
        cost, move_type = alignment_cost(expected_test, actual_test)
        fitness = fitness_from_cost(cost)

    conformance_results[pid] = {
        'hadm_id':       pid,
        'topo_focus':    initial_state['Topographic_Focus'],
        'systemic_stress': initial_state['Systemic_Stress'],
        'expected_test': expected_test,
        'actual_test':   actual_test,
        'move_type':     move_type,
        'alignment_cost': cost,
        'fitness':       fitness,
    }

print(f"Conformance computed for {len(conformance_results)} patients.\n")


# ──────────────────────────────────────────────────────────────
# 5. Summary Statistics
# ──────────────────────────────────────────────────────────────
from collections import Counter

move_counts   = Counter(r['move_type'] for r in conformance_results.values())
fit_scores    = [r['fitness'] for r in conformance_results.values() if r['fitness'] is not None]
mean_fitness  = sum(fit_scores) / len(fit_scores)

print("=== Conformance Summary ===")
print(f"  Mean fitness score : {mean_fitness:.3f}")
print(f"  Sync (match)       : {move_counts['sync']}  "
      f"({move_counts['sync']/len(conformance_results):.1%})")
print(f"  Wrong modality     : {move_counts['wrong_mod']}  "
      f"({move_counts['wrong_mod']/len(conformance_results):.1%})")
print(f"  Wrong region       : {move_counts['wrong_region']}  "
      f"({move_counts['wrong_region']/len(conformance_results):.1%})")
print(f"  No radiology       : {move_counts['no_radiology']}")

# Expected test distribution (what rubric recommends for this cohort)
print("\n=== Rubric Recommendations Distribution ===")
expected_dist = Counter(r['expected_test'] for r in conformance_results.values())
for test, cnt in expected_dist.most_common():
    print(f"  {test:<25} {cnt:4d}  ({cnt/len(conformance_results):.1%})")

# Actual first test distribution
print("\n=== Actual First Test Distribution ===")
actual_dist = Counter(r['actual_test'] for r in conformance_results.values() if r['actual_test'])
for test, cnt in actual_dist.most_common():
    print(f"  {test:<25} {cnt:4d}  ({cnt/actual_dist.total():.1%})")



Conformance computed for 648 patients.

=== Conformance Summary ===
  Mean fitness score : 0.595
  Sync (match)       : 272  (42.0%)
  Wrong modality     : 227  (35.0%)
  Wrong region       : 149  (23.0%)
  No radiology       : 0

=== Rubric Recommendations Distribution ===
  Ultrasound_Abdomen         381  (58.8%)
  CT_Abdomen                 206  (31.8%)
  MRCP_Abdomen                36  (5.6%)
  Radiograph_Abdomen          17  (2.6%)
  CT_Chest                     6  (0.9%)
  Radiograph_Chest             2  (0.3%)

=== Actual First Test Distribution ===
  Ultrasound_Abdomen         360  (55.6%)
  Radiograph_Chest           142  (21.9%)
  CT_Abdomen                  92  (14.2%)
  Radiograph_Abdomen          17  (2.6%)
  MRCP_Abdomen                17  (2.6%)
  CT_Chest                     7  (1.1%)
  ERCP_Abdomen                 6  (0.9%)
  CT_Head                      2  (0.3%)
  MRI_Abdomen                  2  (0.3%)
  Ultrasound_Neck              1  (0.2%)
  CTU_Abdomen           

In [37]:
# ──────────────────────────────────────────────────────────────
# 6. Save conformance results
# ──────────────────────────────────────────────────────────────
with open('data_chole/conformance_results_chole.json', 'w') as f:
    json.dump(conformance_results, f, indent=2)

import csv
fieldnames = list(next(iter(conformance_results.values())).keys())
with open('data_chole/conformance_results_chole.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(conformance_results.values())

print("\nSaved to:")
print("  data_chole/conformance_results_chole.json")
print("  data_chole/conformance_results_chole.csv")


Saved to:
  data_chole/conformance_results_chole.json
  data_chole/conformance_results_chole.csv


In [38]:
import json, math, random
from collections import Counter

# ──────────────────────────────────────────────────────────────
# 1. Load previously saved data
# ──────────────────────────────────────────────────────────────
with open('data_chole/patient_features_chole.json') as f:
    patient_features = json.load(f)

with open('data_chole/conformance_results_chole.json') as f:
    conformance_results = json.load(f)

# ──────────────────────────────────────────────────────────────
# 2. Train / Test Split
# ──────────────────────────────────────────────────────────────
random.seed(42)
all_pids = list(patient_features.keys())
random.shuffle(all_pids)

N_TEST   = 50
K_SIMILAR = 10

test_pids  = set(all_pids[:N_TEST])
train_pids = set(all_pids[N_TEST:])

print(f"Test set       : {len(test_pids)} patients")
print(f"Historical set : {len(train_pids)} patients")
print(f"k (similar)    : {K_SIMILAR}")


# ──────────────────────────────────────────────────────────────
# 3. Feature Vectorization
# ──────────────────────────────────────────────────────────────
PAIN_LOCATIONS = [
    'RUQ','RLQ','LLQ','LUQ','Epigastric',
    'Periumbilical','Pelvic','General_Abdomen','Chest','Other'
]
BINARY_FEATURES = [
    'pain_onset','nausea_vomiting','subj_fever','jaundice','radiating_pain',
    'fever_objective','tachycardia','hypotension','peritoneal_signs','murphy_sign',
    'has_malignancy','is_immunosuppressed','recent_hospitalization',
    'recent_procedure','has_diabetes',
]
LAB_FEATURES = [
    'wbc_status','bilirubin_status','alt_status','alkphos_status',
    'lipase_status','creatinine_status','inr_status','lactate_status',
]

FEATURE_NAMES = (
    [f'loc_{l}' for l in PAIN_LOCATIONS] +   # one-hot  (10)
    ['age_group_norm']                       +   # ordinal   (1)
    BINARY_FEATURES                          +   # binary   (15)
    LAB_FEATURES                                 # ternary   (8)
)  # total: 34 dims

def featurize(fd):
    vec = []
    loc = fd.get('pain_location', 'General_Abdomen')
    vec += [1.0 if loc == l else 0.0 for l in PAIN_LOCATIONS]   # one-hot
    vec.append(fd.get('age_group', 1) / 2.0)                     # normalize
    vec += [float(fd.get(f, 0)) for f in BINARY_FEATURES]        # 0/1
    vec += [(fd.get(f, 0) + 1) / 2.0 for f in LAB_FEATURES]      # -1/0/1 → 0/0.5/1
    return vec

feature_vectors = {pid: featurize(patient_features[pid]) for pid in all_pids}


# ──────────────────────────────────────────────────────────────
# 4. Objective-specific Weights
#    obj1 (recommendation): pain profile + labs
#    obj2 (risk flagging) : risk flags + severity markers
# ──────────────────────────────────────────────────────────────
def build_weights(objective):
    w = {name: 1.0 for name in FEATURE_NAMES}
    if objective == 'obj1':
        for name in FEATURE_NAMES:
            if name.startswith('loc_'):                          w[name] = 2.0
            elif name in LAB_FEATURES:                           w[name] = 1.5
            elif name in ('peritoneal_signs','murphy_sign',
                          'fever_objective','tachycardia'):      w[name] = 1.5
            elif name in ('has_malignancy','is_immunosuppressed',
                          'recent_hospitalization'):             w[name] = 0.5
    elif objective == 'obj2':
        for name in FEATURE_NAMES:
            if name in ('has_malignancy','is_immunosuppressed',
                        'recent_hospitalization','recent_procedure',
                        'has_diabetes'):                         w[name] = 3.0
            elif name in ('fever_objective','tachycardia',
                          'hypotension','peritoneal_signs'):     w[name] = 2.0
            elif name in ('wbc_status','lactate_status',
                          'creatinine_status','inr_status'):     w[name] = 2.0
            elif name.startswith('loc_'):                        w[name] = 0.5
    return [w[name] for name in FEATURE_NAMES]

W_OBJ1 = build_weights('obj1')
W_OBJ2 = build_weights('obj2')


# ──────────────────────────────────────────────────────────────
# 5. Weighted Cosine Similarity + Retrieval
# ──────────────────────────────────────────────────────────────
def weighted_cosine(v1, v2, weights):
    dot   = sum(w*a*b for w,a,b in zip(weights, v1, v2))
    norm1 = math.sqrt(sum(w*a*a for w,a in zip(weights, v1)))
    norm2 = math.sqrt(sum(w*b*b for w,b in zip(weights, v2)))
    return dot / (norm1 * norm2) if norm1 > 0 and norm2 > 0 else 0.0

def retrieve_similar(pid, k=K_SIMILAR, objective='obj1'):
    weights = W_OBJ1 if objective == 'obj1' else W_OBJ2
    qvec = feature_vectors[pid]
    scores = [(hpid, weighted_cosine(qvec, feature_vectors[hpid], weights))
              for hpid in train_pids]
    scores.sort(key=lambda x: -x[1])
    return scores[:k]


Test set       : 50 patients
Historical set : 598 patients
k (similar)    : 10


In [39]:
# ──────────────────────────────────────────────────────────────
# 6. Objective 1: Personalized Test Recommendation
#
#   For each test patient:
#     - Rubric Engine  → deterministic expected_test from initial state
#     - Data Engine    → weighted vote over similar patients' actual_test
#                        weighted by: similarity_score × (1 + fitness)
#                        (high-similarity + high-fitness neighbors get more say)
#     - Compare        → agreement or conflict
# ──────────────────────────────────────────────────────────────
def obj1_recommend(test_pid, similar):
    """
    Rubric Engine:  deterministic recommendation from test patient's initial state
    Data Engine:    similarity-weighted vote over similar patients' ACTUAL tests
                    (no fitness involvement in voting — empirical data speaks independently)
    Group Fitness:  re-computed for the similar group, as a diagnostic signal:
                    high  → similar patients tended to follow rubric  (rubric supported)
                    low   → similar patients systematically deviated  (conflict meaningful)
    """
    rubric_rec = conformance_results[test_pid]['expected_test']

    # ── Data Engine: pure similarity-weighted vote ─────────────
    votes        = {}   # test_name → sum of similarity scores
    fitness_list = []   # per-neighbor fitness (collected separately)

    for hpid, sim in similar:
        cr      = conformance_results[hpid]
        actual  = cr['actual_test']
        fitness = cr['fitness'] if cr['fitness'] is not None else 0.0

        fitness_list.append(fitness)     # record but do NOT use in vote weight

        if actual:
            votes[actual] = votes.get(actual, 0) + sim   # weight = similarity only

    # Empirical recommendation = plurality winner
    empirical_rec = max(votes, key=votes.get) if votes else None

    # Normalised vote distribution (%)
    total = sum(votes.values())
    vote_pct = {t: round(v/total*100, 1)
                for t, v in sorted(votes.items(), key=lambda x: -x[1])}

    # ── Group Fitness: independent signal ──────────────────────
    # How much did this similar-patient group conform to the rubric?
    # (rubric applied to THEIR own initial state, not the test patient's)
    group_fitness = round(sum(fitness_list)/len(fitness_list), 3) if fitness_list else None

    # ── Conflict Layer ─────────────────────────────────────────
    agreement = (empirical_rec == rubric_rec) if empirical_rec else None

    # Interpret the combination of agreement and group_fitness:
    if agreement is None:
        signal = 'no_data'
    elif agreement and group_fitness >= 0.7:
        signal = 'strong_rubric_support'    # empirical confirms rubric, group compliant
    elif agreement and group_fitness < 0.7:
        signal = 'weak_rubric_support'      # same recommendation but group often deviated
    elif not agreement and group_fitness < 0.5:
        signal = 'empirical_override'       # similar pts systematically chose differently
    else:
        signal = 'conflict'                 # disagreement despite reasonable compliance

    return {
        'rubric_rec':      rubric_rec,
        'empirical_rec':   empirical_rec,
        'vote_dist':       vote_pct,
        'group_fitness':   group_fitness,   # rubric compliance of similar-patient group
        'agreement':       agreement,
        'signal':          signal,          # interpretation for clinician
    }



In [40]:
# ──────────────────────────────────────────────────────────────
# 7. Run on test set
# ──────────────────────────────────────────────────────────────
test_results = {}
for pid in test_pids:
    similar = retrieve_similar(pid, k=K_SIMILAR, objective='obj1')
    rec     = obj1_recommend(pid, similar)
    test_results[pid] = {
        'hadm_id':       pid,
        'top_similar':   [(hpid, round(sim, 4)) for hpid, sim in similar],
        **rec,
    }


In [41]:
from collections import Counter

signal_counts = Counter(r['signal'] for r in test_results.values())
mean_gf = sum(r['group_fitness'] for r in test_results.values()
              if r['group_fitness']) / N_TEST

print(f"\n{'='*55}")
print(f"Objective 1 — Personalized Recommendation (N={N_TEST}, k={K_SIMILAR})")
print(f"{'='*55}")
print(f"  Mean group fitness              : {mean_gf:.3f}")
print(f"  Signal distribution:")
for sig, cnt in signal_counts.most_common():
    print(f"    {sig:<28} {cnt:3d}  ({cnt/N_TEST:.0%})")


Objective 1 — Personalized Recommendation (N=50, k=10)
  Mean group fitness              : 0.649
  Signal distribution:
    conflict                      19  (38%)
    weak_rubric_support           16  (32%)
    strong_rubric_support         13  (26%)
    empirical_override             2  (4%)


In [44]:
def inspect_patient(test_pid, test_results, conformance_results,
                    patient_features, trajectories, k=K_SIMILAR):
    r   = test_results[test_pid]
    pf  = patient_features[test_pid]
    cr  = conformance_results[test_pid]
    s0  = trajectories[test_pid][0]   # initial state used by rubric

    print(f"\n{'='*60}")
    print(f"TEST PATIENT: {test_pid}")
    print(f"{'='*60}")

    # ── Test patient profile ──────────────────────────────────
    print("\n[Initial State — Rubric Input]")
    print(f"  Topographic Focus  : {s0['Topographic_Focus']}")
    print(f"  Systemic Stress    : {s0['Systemic_Stress']}")
    print(f"  Organ Stress       : Liver={s0['Organ_Stress_Index']['Liver']}, "
          f"Pancreas={s0['Organ_Stress_Index']['Pancreas']}, "
          f"Kidney={s0['Organ_Stress_Index']['Kidney']}")
    print(f"  Structural Findings: {s0['Structural_Findings']}")

    print("\n[Patient Features — Retrieval Input]")
    print(f"  pain_location      : {pf['pain_location']}")
    print(f"  pain_onset/NV/fever: {pf['pain_onset']} / {pf['nausea_vomiting']} / {pf['subj_fever']}")
    print(f"  peritoneal/murphy  : {pf['peritoneal_signs']} / {pf['murphy_sign']}")
    print(f"  WBC / Bili / ALT   : {pf['wbc_status']} / {pf['bilirubin_status']} / {pf['alt_status']}")
    print(f"  Risk flags — malignancy:{pf['has_malignancy']}  "
          f"immunosup:{pf['is_immunosuppressed']}  "
          f"recent_hosp:{pf['recent_hospitalization']}  "
          f"diabetes:{pf['has_diabetes']}")

    # ── Rubric vs Empirical ───────────────────────────────────
    print("\n[Recommendation Comparison]")
    print(f"  Rubric recommends  : {r['rubric_rec']}")
    print(f"  Empirical recommends: {r['empirical_rec']}")
    print(f"  Vote distribution  : {r['vote_dist']}")
    print(f"  Group fitness      : {r['group_fitness']}")
    print(f"  Signal             : >>> {r['signal']} <<<")

    # ── Similar patients detail ───────────────────────────────
    print(f"\n[Top-{k} Similar Patients]")
    print(f"  {'hadm_id':<12} {'sim':>6}  {'actual_test':<25} "
          f"{'expected_test':<25} {'fitness':>7}  move_type")
    print(f"  {'-'*95}")
    for hpid, sim in r['top_similar']:
        hcr = conformance_results[hpid]
        print(f"  {hpid:<12} {sim:>6.4f}  {str(hcr['actual_test']):<25} "
              f"{str(hcr['expected_test']):<25} "
              f"{str(hcr['fitness']):>7}  {hcr['move_type']}")


# ── Usage ─────────────────────────────────────────────────────
# Pick one from each signal category to compare:

# View by signal type
by_signal = {}
for pid, r in test_results.items():
    by_signal.setdefault(r['signal'], []).append(pid)

print("Patients by signal:")
for sig, pids in by_signal.items():
    print(f"  {sig}: {pids[:5]}")   # show first 5 pid per category

Patients by signal:
  weak_rubric_support: ['24902996', '21219598', '29321309', '24908797', '28417950']
  conflict: ['26747261', '23285325', '27770296', '29990056', '21210624']
  strong_rubric_support: ['20690577', '25916880', '23680828', '23968056', '24636219']
  empirical_override: ['28364578', '25785142']


In [45]:
# Example: pick one patient from each signal category
for signal in ['strong_rubric_support', 'conflict', 'empirical_override']:
    if by_signal.get(signal):
        inspect_patient(
            by_signal[signal][0],
            test_results, conformance_results,
            patient_features, trajectories
        )


TEST PATIENT: 20690577

[Initial State — Rubric Input]
  Topographic Focus  : RUQ
  Systemic Stress    : 0.2
  Organ Stress       : Liver=0.5, Pancreas=0.0, Kidney=0.0
  Structural Findings: {'stone_detected': 0, 'wall_thickening': 0, 'fluid_collection': 0, 'organ_dilation': 0}

[Patient Features — Retrieval Input]
  pain_location      : RUQ
  pain_onset/NV/fever: 0 / 1 / 1
  peritoneal/murphy  : 0 / 0
  WBC / Bili / ALT   : 0 / 0 / 1
  Risk flags — malignancy:1  immunosup:0  recent_hosp:0  diabetes:0

[Recommendation Comparison]
  Rubric recommends  : Ultrasound_Abdomen
  Empirical recommends: Ultrasound_Abdomen
  Vote distribution  : {'Ultrasound_Abdomen': 69.4, 'Radiograph_Chest': 20.5, 'CT_Abdomen': 10.1}
  Group fitness      : 0.75
  Signal             : >>> strong_rubric_support <<<

[Top-10 Similar Patients]
  hadm_id         sim  actual_test               expected_test             fitness  move_type
  ----------------------------------------------------------------------------

### Objective 2: Risk control

In [46]:
# key: 关键词（在 ICD Diagnosis 列表中搜索）
# value: 应补充的影像检查

SERIOUS_CONDITION_MAP = {
    # 胆囊炎严重并发症
    'perforation':         ['CT_Abdomen'],
    'gangrenous':          ['CT_Abdomen'],
    'emphysematous':       ['CT_Abdomen'],
    'cholangitis':         ['MRCP_Abdomen'],
    'choledocholithiasis': ['MRCP_Abdomen'],
    'liver abscess':       ['CT_Abdomen'],
    'biliary obstruction': ['MRCP_Abdomen'],

    # 全身性严重状态
    'sepsis':              ['CT_Abdomen'],    # 查感染源
    'septic shock':        ['CT_Abdomen'],

    # 恶性肿瘤背景
    'carcinoma':           ['MRI_Abdomen'],
    'malignancy':          ['MRI_Abdomen'],
    'tumor':               ['MRI_Abdomen'],

    # 其他四种病理的严重型
    'perforated appendicitis': ['CT_Abdomen'],
    'peritonitis':         ['CT_Abdomen'],
    'necrotizing':         ['CT_Abdomen'],   # necrotizing pancreatitis
    'pancreatitis':        ['CT_Abdomen'],   # 意外的胰腺炎
}

# 直接风险规则（来自 test 患者自身 features）
DIRECT_RISK_RULES = [
    # condition dict → additional tests
    {'if': {'is_immunosuppressed': 1},
     'then': ['CT_Abdomen'],
     'reason': 'immunosuppressed: atypical infection risk'},

    {'if': {'has_malignancy': 1},
     'then': ['MRI_Abdomen'],
     'reason': 'malignancy: tumor-related biliary obstruction'},

    {'if': {'tachycardia': 1, 'fever_objective': 1, 'wbc_status': 1},
     'then': ['CT_Abdomen'],
     'reason': 'sepsis triad: tachycardia + fever + elevated WBC'},

    {'if': {'hypotension': 1},
     'then': ['CT_Abdomen'],
     'reason': 'hemodynamic compromise'},

    {'if': {'inr_status': 1, 'bilirubin_status': 1},
     'then': ['MRCP_Abdomen'],
     'reason': 'coagulopathy + hyperbilirubinemia: biliary obstruction risk'},
]

In [47]:
import ast

# Load raw CSV for ICD diagnoses
with open('raw_data/cholecystitis_hadm_info_first_diag.csv') as f:
    reader = csv.DictReader(f)
    raw_rows_all = {row['hadm_id']: row for row in reader}

def get_icd_diagnoses(pid):
    """Return list of ICD diagnosis strings for a patient."""
    raw = raw_rows_all.get(str(pid), {}).get('ICD Diagnosis', '[]')
    try:
        return [d.lower() for d in ast.literal_eval(raw)]
    except:
        return []

def flag_serious_conditions(icd_list):
    """Check ICD diagnosis list against SERIOUS_CONDITION_MAP."""
    flagged = {}   # condition_keyword → additional tests
    for keyword, tests in SERIOUS_CONDITION_MAP.items():
        if any(keyword in diag for diag in icd_list):
            flagged[keyword] = tests
    return flagged

def apply_direct_rules(pf):
    """Apply direct risk rules from test patient's own features."""
    triggered = []
    for rule in DIRECT_RISK_RULES:
        if all(pf.get(k) == v for k, v in rule['if'].items()):
            triggered.append({'reason': rule['reason'], 'tests': rule['then']})
    return triggered

def obj2_risk_flagging(test_pid, obj1_rec, k=K_SIMILAR):
    pf = patient_features[test_pid]

    # ── Layer A: Similar-patient risk flagging ────────────────
    similar = retrieve_similar(test_pid, k=k, objective='obj2')

    layer_a_flags = {}   # pid → {keyword: tests}
    for hpid, sim in similar:
        icd_list = get_icd_diagnoses(hpid)
        conditions = flag_serious_conditions(icd_list)
        if conditions:
            layer_a_flags[hpid] = {'similarity': sim, 'conditions': conditions}

    # Aggregate: collect all additional tests from flagged similar patients
    layer_a_tests = set()
    for hpid, info in layer_a_flags.items():
        for tests in info['conditions'].values():
            layer_a_tests.update(tests)

    # ── Layer B: Direct patient-level risk rules ──────────────
    layer_b_triggers = apply_direct_rules(pf)
    layer_b_tests = set()
    for t in layer_b_triggers:
        layer_b_tests.update(t['tests'])

    # ── Combined: remove tests already in obj1 recommendation ─
    all_risk_tests = layer_a_tests | layer_b_tests
    additional_tests = all_risk_tests - {obj1_rec}   # don't duplicate obj1

    return {
        'hadm_id':           test_pid,
        'obj1_recommendation': obj1_rec,
        'layer_a_flagged_patients': len(layer_a_flags),
        'layer_a_conditions': {h: i['conditions'] for h, i in layer_a_flags.items()},
        'layer_a_tests':     list(layer_a_tests),
        'layer_b_triggers':  layer_b_triggers,
        'layer_b_tests':     list(layer_b_tests),
        'risk_additional_tests': list(additional_tests),   # final output
        'any_risk_flagged':  len(additional_tests) > 0,
    }

# Run on test set
obj2_results = {}
for pid in test_pids:
    obj1_rec = test_results[pid]['rubric_rec']   # or 'final_recommendation'
    obj2_results[pid] = obj2_risk_flagging(pid, obj1_rec)

# Summary
n_flagged = sum(1 for r in obj2_results.values() if r['any_risk_flagged'])
n_layer_a = sum(1 for r in obj2_results.values() if r['layer_a_flagged_patients'] > 0)
n_layer_b = sum(1 for r in obj2_results.values() if r['layer_b_triggers'])

print(f"\n{'='*50}")
print(f"Objective 2 — Risk Flagging (N={N_TEST})")
print(f"{'='*50}")
print(f"  Any risk test added      : {n_flagged} ({n_flagged/N_TEST:.0%})")
print(f"  Triggered by Layer A     : {n_layer_a} ({n_layer_a/N_TEST:.0%})")
print(f"  Triggered by Layer B     : {n_layer_b} ({n_layer_b/N_TEST:.0%})")


Objective 2 — Risk Flagging (N=50)
  Any risk test added      : 28 (56%)
  Triggered by Layer A     : 31 (62%)
  Triggered by Layer B     : 8 (16%)


In [48]:
def inspect_obj2(test_pid, obj2_results, obj1_results, patient_features,
                 conformance_results, raw_rows_all, k=K_SIMILAR):

    r2  = obj2_results[test_pid]
    r1  = obj1_results[test_pid]
    pf  = patient_features[test_pid]

    print(f"\n{'='*65}")
    print(f"OBJECTIVE 2 — RISK FLAGGING:  Patient {test_pid}")
    print(f"{'='*65}")

    # ── Obj1 recommendation (context) ────────────────────────
    print(f"\n[Obj1 Recommendation — context]")
    print(f"  Rubric rec   : {r1['rubric_rec']}")
    print(f"  Empirical rec: {r1['empirical_rec']}")
    print(f"  Signal       : {r1['signal']}")

    # ── Test patient's own risk features ─────────────────────
    print(f"\n[Test Patient Risk Features]")
    risk_feats = ['has_malignancy','is_immunosuppressed','recent_hospitalization',
                  'recent_procedure','has_diabetes','fever_objective','tachycardia',
                  'hypotension','wbc_status','lactate_status','inr_status',
                  'bilirubin_status','creatinine_status']
    for f in risk_feats:
        val = pf.get(f, 0)
        flag = ' ◄ HIGH' if (val == 1 and f not in ('wbc_status','lactate_status',
                             'inr_status','bilirubin_status','creatinine_status')) \
               or (val != 0 and f in ('wbc_status','lactate_status',
                   'inr_status','bilirubin_status','creatinine_status')) else ''
        print(f"  {f:<30} {val}{flag}")

    # ── Layer B: Direct rules triggered ──────────────────────
    print(f"\n[Layer B — Direct Risk Rules]")
    if r2['layer_b_triggers']:
        for t in r2['layer_b_triggers']:
            print(f"  TRIGGERED: {t['reason']}")
            print(f"    → Additional tests: {t['tests']}")
    else:
        print("  No direct risk rules triggered.")

    # ── Layer A: Similar-patient flags ────────────────────────
    print(f"\n[Layer A — Similar-Patient Risk Flags]")
    similar_obj2 = retrieve_similar(test_pid, k=k, objective='obj2')

    print(f"  {'hadm_id':<12} {'sim':>6}  {'serious conditions found':<35}  ICD snippets")
    print(f"  {'-'*90}")

    for hpid, sim in similar_obj2:
        icd_list = get_icd_diagnoses(hpid)
        conditions = flag_serious_conditions(icd_list)

        # Show matching ICD strings
        matched_icds = []
        for keyword in conditions:
            matched_icds += [d for d in icd_list if keyword in d]

        cond_str = ', '.join(conditions.keys()) if conditions else '—'
        icd_str  = ' | '.join(matched_icds[:2]) if matched_icds else ''   # show max 2

        marker = ' ◄' if conditions else ''
        print(f"  {hpid:<12} {sim:>6.4f}  {cond_str:<35}  {icd_str[:45]}{marker}")

    n_flagged_pts = len(r2['layer_a_conditions'])
    print(f"\n  → {n_flagged_pts}/{k} similar patients had serious conditions")

    # ── Combined result ───────────────────────────────────────
    print(f"\n[FINAL RISK OUTPUT]")
    print(f"  Obj1 recommendation      : {r2['obj1_recommendation']}")
    if r2['risk_additional_tests']:
        print(f"  Risk additional tests    : {r2['risk_additional_tests']}")
        print(f"  (Layer A: {r2['layer_a_tests']})")
        print(f"   Layer B: {r2['layer_b_tests']})")
    else:
        print(f"  Risk additional tests    : None — no high-risk signals detected")

    print(f"\n  Full recommendation bundle:")
    full_bundle = list({r2['obj1_recommendation']} | set(r2['risk_additional_tests']))
    for i, test in enumerate(full_bundle, 1):
        tag = '[OBJ1]' if test == r2['obj1_recommendation'] else '[RISK]'
        print(f"    {i}. {tag}  {test}")

In [50]:
# 先看一个有风险标记的患者
flagged_pids = [pid for pid, r in obj2_results.items() if r['any_risk_flagged']]
not_flagged  = [pid for pid, r in obj2_results.items() if not r['any_risk_flagged']]

print(f"Patients with risk flags: {len(flagged_pids)}")
print(f"  IDs: {flagged_pids[:5]}")

# 查看一个有 flag 的
if flagged_pids:
    inspect_obj2(flagged_pids[0], obj2_results, test_results,
                 patient_features, conformance_results, raw_rows_all)


Patients with risk flags: 28
  IDs: ['24902996', '21219598', '29321309', '20690577', '24908797']

OBJECTIVE 2 — RISK FLAGGING:  Patient 24902996

[Obj1 Recommendation — context]
  Rubric rec   : Ultrasound_Abdomen
  Empirical rec: Ultrasound_Abdomen
  Signal       : weak_rubric_support

[Test Patient Risk Features]
  has_malignancy                 0
  is_immunosuppressed            0
  recent_hospitalization         0
  recent_procedure               0
  has_diabetes                   0
  fever_objective                0
  tachycardia                    0
  hypotension                    0
  wbc_status                     0
  lactate_status                 0
  inr_status                     1 ◄ HIGH
  bilirubin_status               0
  creatinine_status              0

[Layer B — Direct Risk Rules]
  No direct risk rules triggered.

[Layer A — Similar-Patient Risk Flags]
  hadm_id         sim  serious conditions found             ICD snippets
  -----------------------------------------

In [52]:
# 查看一个没有 flag 的，作为对比
if not_flagged:
    inspect_obj2(not_flagged[0], obj2_results, test_results,
                 patient_features, conformance_results, raw_rows_all)


OBJECTIVE 2 — RISK FLAGGING:  Patient 26747261

[Obj1 Recommendation — context]
  Rubric rec   : CT_Abdomen
  Empirical rec: Ultrasound_Abdomen
  Signal       : conflict

[Test Patient Risk Features]
  has_malignancy                 0
  is_immunosuppressed            0
  recent_hospitalization         0
  recent_procedure               0
  has_diabetes                   0
  fever_objective                0
  tachycardia                    0
  hypotension                    0
  wbc_status                     0
  lactate_status                 0
  inr_status                     0
  bilirubin_status               1 ◄ HIGH
  creatinine_status              0

[Layer B — Direct Risk Rules]
  No direct risk rules triggered.

[Layer A — Similar-Patient Risk Flags]
  hadm_id         sim  serious conditions found             ICD snippets
  ------------------------------------------------------------------------------------------
  24642301     0.9220  —                                    
  204